In [1]:
# Install Klygo from the v2-release branch
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

# Mining Curation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/mining/Document_Mining_Curation.ipynb)

Perform deterministic clustering, diverse sample selection across 9 strategies, and class imbalance mitigation.

## Jump to an API

- [mining.cluster](#mining.cluster)
- [mining.select](#mining.select)
- [mining.balance](#mining.balance)
- [mining.ClusterResult](#mining.ClusterResult)

### mining.cluster

Group images using histograms, HOG, quality, boxes, and class presence.

In [2]:
from klygo import archive, datasets, files, mining

workspace = files.join("artifacts", "tutorials", "mining")
flat_dir = files.join(workspace, "flat_coco8")
if not files.exists(files.join(flat_dir, "data.yaml")):
    files.mkdir(workspace)
    zip_path = files.download("https://github.com/ultralytics/assets/releases/download/v0.0.0/coco8.zip", workspace, overwrite=True, verbose=False)
    yaml_path = files.download("https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/cfg/datasets/coco8.yaml", workspace, overwrite=True, verbose=False)
    raw_dir = files.join(workspace, "raw_coco8")
    archive.extract(zip_path, raw_dir, overwrite=True, verbose=False)
    files.copy(yaml_path, files.join(raw_dir, "coco8.yaml"), overwrite=True)
    datasets.convert(
        source=raw_dir,
        target=flat_dir,
        tree={
            "images": ["coco8/images/train", "coco8/images/val"],
            "labels": ["coco8/labels/train", "coco8/labels/val"],
            "metadata": "coco8.yaml",
        },
        on_conflict="rename",
        overwrite=True,
    )

data = datasets.load(flat_dir, format="yolo", verbose=False)
print(f"Loaded dataset: {len(data)} images, {data.total_objects} boxes, {len(data.names)} classes")

clusters = mining.cluster(data, clusters=min(3, len(data)), seed=42)
clusters.show()
print("Cluster representatives:", clusters.representatives)

Loaded dataset: 8 images, 30 boxes, 12 classes
           ClusterResult           
┌─────────┬──────┬────────────────┐
│ Cluster │ Size │ Representative │
├─────────┼──────┼────────────────┤
│ 0       │ 4    │ 4              │
│ 1       │ 3    │ 3              │
│ 2       │ 1    │ 0              │
└─────────┴──────┴────────────────┘
Cluster representatives: {0: 4, 1: 3, 2: 0}


### mining.select

Select whole images using all 9 strategies: `random`, `weighted`, `quality`, `rare_first`, `diversity`, `boxes`, `quota`, `coverage`, and `class`.

In [3]:
scores = mining.score(data)
strategies = {
    "random":     mining.select(data, method="random", limit=4, seed=42),
    "weighted":   mining.select(data, method="weighted", scores=scores, limit=4),
    "quality":    mining.select(data, method="quality", scores=scores, threshold=0.3, limit=4),
    "rare_first": mining.select(data, method="rare_first", scores=scores, limit=4),
    "diversity":  mining.select(data, method="diversity", cluster_result=clusters, per_cluster=2, limit=4),
    "boxes":      mining.select(data, method="boxes", min_boxes=2, max_boxes=20, limit=4),
    "quota":      mining.select(data, method="quota", per_class=1, limit=4),
    "coverage":   mining.select(data, method="coverage", per_class=1, limit=4),
    "class":      mining.select(data, method="class", classes=["person"], match="any", limit=4),
}
for name, sub in strategies.items():
    print(f"method={name!r:<12} -> {len(sub)} images ({sub.total_objects} boxes)")

method='random'     -> 4 images (20 boxes)
method='weighted'   -> 4 images (12 boxes)
method='quality'    -> 4 images (12 boxes)
method='rare_first' -> 4 images (12 boxes)
method='diversity'  -> 4 images (7 boxes)
method='boxes'      -> 4 images (14 boxes)
method='quota'      -> 4 images (21 boxes)
method='coverage'   -> 4 images (21 boxes)
method='class'      -> 3 images (16 boxes)


### mining.balance

Reduce imbalance without creating duplicate frame identities (`target='min'`, `'median'`, `'mean'`, or integer).

In [4]:
for target_mode in ("min", "median", "mean"):
    balanced = mining.balance(data, target=target_mode, count_by="images", seed=42)
    print(f"balance(target={target_mode!r:<8}) -> {len(balanced)}/{len(data)} images")

balance(target='min'   ) -> 8/8 images
balance(target='median') -> 8/8 images
balance(target='mean'  ) -> 8/8 images


### mining.ClusterResult

Cluster assignments, groups, and representatives returned by `cluster`.

In [5]:
cluster_path = clusters.save("artifacts/tutorials/mining/clusters_report.json")
print("Cluster groups:", clusters.groups)
print("Saved ClusterResult to:", cluster_path)

Cluster groups: {0: [2, 4, 5, 6], 1: [1, 3, 7], 2: [0]}
Saved ClusterResult to: None


## Put it together

Cluster images by visual features, select diverse representatives, and balance class distributions.

In [6]:
clusters = mining.cluster(data, clusters=min(3, len(data)), seed=42)
curated = mining.select(data, method="diversity", cluster_result=clusters, per_cluster=2, limit=5)
balanced = mining.balance(curated, target="median", seed=42)
out_dir = balanced.export("artifacts/tutorials/mining/curated_balanced", format="yolo", overwrite=True)
print(f"Curated and balanced dataset ({len(balanced)} images) exported to: {out_dir}")

Curated and balanced dataset (5 images) exported to: None


## Remember

1. `cluster` uses deterministic K-Means over normalized visual and annotation features.
2. `select` supports 9 curation strategies: random, weighted, quality, rare_first, diversity, boxes, quota, coverage, and class.
3. `balance` reduces class imbalance without synthesizing or duplicating images.